# Loading and Inspecting fNIRS Recordings

This notebook shows how to load an fNIRS recording from a SNIRF file and perform a first inspection of the data. Because Cedalion uses the vendor-neutral [SNIRF standard](https://github.com/fNIRS/snirf), the loading step is identical regardless of which device the data were recorded with — the same two lines of code work for NIRx, Artinis, Gowerlabs, Kernel, and any other SNIRF-compatible system.

After loading, we walk through four inspection steps that are useful for any new dataset:

1. **Channel statistics** — distribution of source–detector distances and signal quality (SNR).
2. **Montage** — where are the optodes placed on the scalp?
3. **Sensitivity profile** — which brain regions are covered by this montage?
4. **Time series** — does the raw signal look reasonable?

**Prerequisites:** familiarity with the [Recording container](11_recording_container.ipynb) and [Cedalion data structures](10_xarray_datastructs_fnirs.ipynb).

**Going deeper:** links to more detailed notebooks are provided at each step.

In [ ]:
# This cells setups the environment when executed in Google Colab.
try:
    import google.colab
    !curl -s https://raw.githubusercontent.com/ibs-lab/cedalion/dev/scripts/colab_setup.py -o colab_setup.py
    # Select branch with --branch "branch name" (default is "dev")
    %run colab_setup.py
except ImportError:
    pass

In [ ]:
# set this flag to True to enable interactive 3D plots
INTERACTIVE_PLOTS = False

In [ ]:
import cedalion.dot
import cedalion.io
import cedalion.nirs
import cedalion.vis.anatomy
import cedalion.vis.blocks as vbx
import cedalion.sigproc.quality as qlty
import pyvista as pv
import matplotlib.pyplot as plt
from cedalion.geometry.landmarks import normalize_landmarks_labels
import cedalion.data
import numpy as np

if INTERACTIVE_PLOTS:
    pv.set_jupyter_backend("server")
else:
    pv.set_jupyter_backend("static")

## Head model

Several of the inspection steps below (montage visualisation, sensitivity profiling) require a head model — a pair of 3-D surface meshes representing the scalp and the cortex together with standard landmark positions. We load it once here and reuse it throughout.

Cedalion ships pre-built head models for the standard atlases **Colin27** and **ICBM152**. For subject-specific analyses you can build a custom head model from an individual MRI; see the [Head Models: MRI Segmentation and TwoSurfaceHeadModel Notebook](../head_models/44_head_models.ipynb) and  [Image Reconstruction Notebook](../head_models/40_image_reconstruction.ipynb) for details.

In [ ]:
# Load the Colin27 atlas head model in voxel (ijk) space, then transform it
# into RAS world-space coordinates (mm) for visualisation and geometry operations.
head_ijk = cedalion.dot.get_standard_headmodel("colin27")
head_ras = head_ijk.apply_transform(head_ijk.t_ijk2ras)

## Loading and inspecting recordings

Any SNIRF file can be loaded with a single function call, regardless of vendor:

```python
rec = cedalion.io.read_snirf("/path/to/your/recording.snirf")
```

This returns a `Recording` object that holds the raw amplitude time series, optode geometry, stimulus table, and auxiliary channels. The four inspection steps below — channel statistics, montage, sensitivity profile, and time series — are applied identically to each example dataset.

The following sections walk through example datasets from four different vendors:
1. **NIRx** (NIRSport2) 
2. **Gowerlabs** (LUMO) 
3. **Artinis** (Brite mk2) 
4. **Kernel** (Flow)

---
### 1. NIRx

[NIRx](https://nirx.net/) provides tabletop and wearable devices with freely configurable otpodes. The example here uses our lab's own fingertapping example from a HD-Probe that is also shown in other example notebooks.

In [ ]:
# Load the example NSP2 dataset (downloaded automatically on first use).
# To use your own data, replace this line with:
#   rec = cedalion.io.load_recording("/path/to/your/recording.snirf")
rec = cedalion.data.get_fingertappingDOT()

#### Step 1 — Channel statistics

Before looking at the signal, it is useful to characterise the channel geometry and raw signal quality.

- **Source–detector distance** determines measurement depth: short-separation channels (< 1 cm) primarily sample the scalp; long-separation channels (2.5–4.5 cm) are sensitive to cortical haemodynamics. Above 3.5 cm the sensitivity to cortical haemodynamics improves further, however the amount of received light and thus SNR drastically decreases. 
- **Signal-to-noise ratio (SNR)** is a fast first-pass quality check. Channels with very low SNR (typically caused by poor optode contact) should be excluded before further analysis.

> For a full quality assessment workflow — including scalp coupling index (SCI), peak-power ratio, and channel pruning — see the [signal quality notebook](../signal_quality/23_pruning.ipynb).

In [ ]:
# calculate the distances for all channels in the recording
distances = cedalion.nirs.channel_distances(rec["amp"], rec.geo3d)
channel_count = len(distances)

# calculate SNR for all channels
snr, _ = qlty.snr(rec["amp"])

# create the plots...
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
# ... a histogram of SNR ...
axes[0].hist(distances, bins=50, alpha=0.7, color="steelblue")
axes[0].set_title(f"Channel distances ({channel_count} channels)")
axes[0].set_xlabel("Distance (mm)")
axes[0].set_ylabel("Count")
axes[0].grid(True)
#... and a scatter plot of SNR vs distance
axes[1].scatter(distances, snr.sel(wavelength=850), alpha=0.7, color="coral", s=2)
axes[1].set_title("SNR vs Channel Distance")
axes[1].set_xlabel("Distance (mm)")
axes[1].set_ylabel("SNR")
axes[1].grid(True)
# show the plots
fig.tight_layout()
plt.show()


Depending on the measurement setup and vendor all possible source–detector combinations are stored in the SNIRF file, many of which can be too far apart to carry a useful cortical signal. For this example, we retain only channels with a source–detector distance below 3.5 cm.

In [ ]:
# generate the distance mask
ch_mask = cedalion.nirs.channel_distances(rec["amp"], rec.geo3d) < 3.5 * cedalion.units.cm

# apply the mask to the data
amp = rec["amp"].sel(channel=ch_mask)

# print the number of channels that pass the mask relative to the total number
print(f"Channels passing distance mask: {ch_mask.sum().item()} / {len(ch_mask)}")

#### Step 2 — Montage visualisation

Visualising the probe montage on a head surface confirms that the optodes are in anatomically plausible positions and gives an immediate sense of the cortical coverage.

We first plot the raw digitised positions reported in the SNIRF file, then register them to the Colin27 atlas scalp surface. Registration involves two sub-steps:

1. **Landmark normalisation** — map vendor-specific landmark labels (e.g. `"Nz"`, `"nz"`, `"nasion"`) to a common naming convention so they align with the atlas landmarks.
2. **Scalp snapping** — rigidly align the digitised positions to the atlas and project each optode onto the nearest point on the scalp mesh.

> To learn more about probe registration and co-registration using photogrammetry, see the [Photogrammetric Co-Registration Notebook](../head_models/41_photogrammetric_optode_coregistration.ipynb).

In [ ]:
# Quick 3-D view of the optode positions as stored in the SNIRF file,
# without any registration to a head model.
cedalion.vis.anatomy.plot_montage3D(amp, rec.geo3d)

In [ ]:
# Normalise landmark labels so they match the atlas convention, then
# rigidly align and snap the optode positions onto the Colin27 scalp surface.
geo3d = normalize_landmarks_labels(rec.geo3d)
geo3d_snapped = head_ras.align_and_snap_to_scalp(geo3d)

# Render the scalp (semi-transparent) and cortex together with the registered optodes.
plotter = pv.Plotter()
vbx.plot_surface(plotter, head_ras.scalp, opacity=0.1)
vbx.plot_surface(plotter, head_ras.brain, color="w")
vbx.plot_labeled_points(plotter, geo3d_snapped)
plotter.show()

#### Step 3 — Sensitivity profile

The sensitivity matrix (also called the Jacobian or *A* matrix) quantifies how much each cortical voxel contributes to each measured channel. Visualising the maximum sensitivity across all channels gives an intuitive map of the cortical area covered by the montage and reveals potential blind spots.

Computing the sensitivity matrix requires running a photon transport simulation (Monte Carlo or FEM), which is computationally expensive. Here we load a pre-computed sensitivity matrix for this dataset. To learn how to compute your own, see the [image reconstruction notebook](../head_models/40_image_reconstruction.ipynb).

In [ ]:
# Load a pre-computed sensitivity matrix for this dataset at the Colin27 atlas.
Adot = cedalion.data.get_precomputed_sensitivity("fingertappingDOT", "colin27")

# For a spatial overview, take the maximum sensitivity across all channels at 850 nm.
# Restricting to brain vertices avoids scalp contributions dominating the colour scale.
# Log10 is applied because sensitivity varies over several orders of magnitude with depth.
sensitivity = Adot.sel(vertex=Adot.is_brain, wavelength=850).max("channel")

plotter = pv.Plotter()
vbx.plot_surface(plotter, head_ras.scalp, color="w", opacity=0.1)
vbx.plot_surface(plotter, head_ras.brain, color=np.log10(np.clip(sensitivity, 1e-3, 1e1)), cmap="jet")
vbx.plot_labeled_points(plotter, geo3d_snapped)
plotter.show()

#### Step 4 — Time series

A quick look at the raw amplitude time series for a single channel confirms that the signal is present and physiologically plausible. Things to look for:

- A clear cardiac pulse (~1 Hz oscillation visible in a short window).
- No flat sections or sudden jumps that would indicate a saturated or disconnected optode.
- Stimulus markers aligned with the expected task timings.

> For detailed time series visualisation across many channels — including butterfly plots, quality overlays, and epoch-locked averages — see the [Visualization Examples Notebook](../plots_visualization/12_plots_example.ipynb).

In [ ]:
ch = amp.channel[0].item()  # pick first channel; replace with a specific channel name if desired

fig, ax = plt.subplots(figsize=(10, 3))
for wl, color in zip(amp.wavelength.values, ["b", "r"]):
    ax.plot(amp.time, amp.sel(channel=ch, wavelength=wl), color=color, label=f"{wl} nm")
ax.set_title(f"Raw amplitude — Ch. {ch}")
ax.set_xlabel("Time / s")
ax.set_ylabel("Intensity / a.u.")
ax.set_xlim(0, 100)
ax.legend()
vbx.plot_stim_markers(ax, rec.stim, y=1)
plt.tight_layout()
plt.show()

---
### 1. Gowerlabs (LUMO)

The Lumo from [Gowerlabs](https://www.gowerlabs.co.uk) is a wearable, modular HD-DOT system. Its tile-based design generates a dense grid of source–detector pairs, including many short-separation channels. Individual optodes cannot be freely configured, but optodes grouped by tiles. Example data here was provided by Gowerlabs.

In [ ]:
# Load the example LUMO dataset (downloaded automatically on first use).
# To use your own data, replace this line with:
#   rec = cedalion.io.load_recording("/path/to/your/recording.snirf")
rec = cedalion.data.get_lumo_testdataset()

#### Step 1 — Channel statistics

Before looking at the signal, it is useful to characterise the channel geometry and raw signal quality.

- **Source–detector distance** determines measurement depth: short-separation channels (< 1 cm) primarily sample the scalp; long-separation channels (2.5–4.5 cm) are sensitive to cortical haemodynamics. Above 3.5 cm the sensitivity to cortical haemodynamics improves further, however the amount of received light and thus SNR drastically decreases. 
- **Signal-to-noise ratio (SNR)** is a fast first-pass quality check. Channels with very low SNR (typically caused by poor optode contact) should be excluded before further analysis.

> For a full quality assessment workflow — including scalp coupling index (SCI), peak-power ratio, and channel pruning — see the [signal quality notebook](../signal_quality/23_pruning.ipynb).

In [ ]:
# calculate the distances for all channels in the recording
distances = cedalion.nirs.channel_distances(rec["amp"], rec.geo3d)
channel_count = len(distances)

# calculate SNR for all channels
snr, _ = qlty.snr(rec["amp"])

# create the plots...
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
# ... a histogram of SNR ...
axes[0].hist(distances, bins=50, alpha=0.7, color="steelblue")
axes[0].set_title(f"Channel distances ({channel_count} channels)")
axes[0].set_xlabel("Distance (mm)")
axes[0].set_ylabel("Count")
axes[0].grid(True)
#... and a scatter plot of SNR vs distance
axes[1].scatter(distances, snr.sel(wavelength=850), alpha=0.7, color="coral", s=2)
axes[1].set_title("SNR vs Channel Distance")
axes[1].set_xlabel("Distance (mm)")
axes[1].set_ylabel("SNR")
axes[1].grid(True)
# show the plots
fig.tight_layout()
plt.show()


Depending on the measurement setup and vendor all possible source–detector combinations are stored in the SNIRF file, many of which can be too far apart to carry a useful cortical signal. For this example, we retain only channels with a source–detector distance below 3.5 cm.

In [ ]:
# generate the distance mask
ch_mask = cedalion.nirs.channel_distances(rec["amp"], rec.geo3d) < 3.5 * cedalion.units.cm

# apply the mask to the data
amp = rec["amp"].sel(channel=ch_mask)

# print the number of channels that pass the mask relative to the total number
print(f"Channels passing distance mask: {ch_mask.sum().item()} / {len(ch_mask)}")

#### Step 2 — Montage visualisation

Visualising the probe montage on a head surface confirms that the optodes are in anatomically plausible positions and gives an immediate sense of the cortical coverage.

We first plot the raw digitised positions reported in the SNIRF file, then register them to the Colin27 atlas scalp surface. Registration involves two sub-steps:

1. **Landmark normalisation** — map vendor-specific landmark labels (e.g. `"Nz"`, `"nz"`, `"nasion"`) to a common naming convention so they align with the atlas landmarks.
2. **Scalp snapping** — rigidly align the digitised positions to the atlas and project each optode onto the nearest point on the scalp mesh.

> To learn more about probe registration and co-registration using photogrammetry, see the [Photogrammetric Co-Registration Notebook](../head_models/41_photogrammetric_optode_coregistration.ipynb).

In [ ]:
# Quick 3-D view of the optode positions as stored in the SNIRF file,
# without any registration to a head model.
cedalion.vis.anatomy.plot_montage3D(amp, rec.geo3d)

In [ ]:
# Normalise landmark labels so they match the atlas convention, then
# rigidly align and snap the optode positions onto the Colin27 scalp surface.
geo3d = normalize_landmarks_labels(rec.geo3d)
geo3d_snapped = head_ras.align_and_snap_to_scalp(geo3d)

# Render the scalp (semi-transparent) and cortex together with the registered optodes.
Plotter = pv.Plotter()
vbx.plot_surface(Plotter, head_ras.scalp, opacity=0.1)
vbx.plot_surface(Plotter, head_ras.brain, color="w")
vbx.plot_labeled_points(Plotter, geo3d_snapped)
Plotter.show()

#### Step 3 — Sensitivity profile

The sensitivity matrix (also called the Jacobian or *A* matrix) quantifies how much each cortical voxel contributes to each measured channel. Visualising the maximum sensitivity across all channels gives an intuitive map of the cortical area covered by the montage and reveals potential blind spots.

Computing the sensitivity matrix requires running a photon transport simulation (Monte Carlo or FEM), which is computationally expensive. Here we load a pre-computed sensitivity matrix for this dataset. To learn how to compute your own, see the [image reconstruction notebook](../head_models/40_image_reconstruction.ipynb).

In [ ]:
# Load a pre-computed sensitivity matrix for this dataset at the Colin27 atlas.
Adot = cedalion.data.get_precomputed_sensitivity("lumo_testdataset", "colin27")

# For a spatial overview, take the maximum sensitivity across all channels at 850 nm.
# Restricting to brain vertices avoids scalp contributions dominating the colour scale.
# Log10 is applied because sensitivity varies over several orders of magnitude with depth.
sensitivity = Adot.sel(vertex=Adot.is_brain, wavelength=850).max("channel")

Plotter = pv.Plotter()
vbx.plot_surface(Plotter, head_ras.scalp, color="w", opacity=0.5)
vbx.plot_surface(Plotter, head_ras.brain, color=np.log10(np.clip(sensitivity, 1e-3, 1e1)), cmap="jet")
vbx.plot_labeled_points(Plotter, geo3d_snapped)
Plotter.show()

#### Step 4 — Time series

A quick look at the raw amplitude time series for a single channel confirms that the signal is present and physiologically plausible. Things to look for:

- A clear cardiac pulse (~1 Hz oscillation visible in a short window).
- No flat sections or sudden jumps that would indicate a saturated or disconnected optode.
- Stimulus markers aligned with the expected task timings.

> For detailed time series visualisation across many channels — including butterfly plots, quality overlays, and epoch-locked averages — see the [Visualization Examples Notebook](../plots_visualization/12_plots_example.ipynb).

In [ ]:
ch = amp.channel[0].item()  # pick first channel; replace with a specific channel name if desired

fig, ax = plt.subplots(figsize=(10, 3))
for wl, color in zip(amp.wavelength.values, ["b", "r"]):
    ax.plot(amp.time, amp.sel(channel=ch, wavelength=wl), color=color, label=f"{wl} nm")
ax.set_title(f"Raw amplitude — Ch. {ch}")
ax.set_xlabel("Time / s")
ax.set_ylabel("Intensity / a.u.")
ax.set_xlim(0, 100)
ax.legend()
vbx.plot_stim_markers(ax, rec.stim, y=1)
plt.tight_layout()
plt.show()

---
### 3. Artinis

[Artinis](https://www.artinis.com/) provides wearable devices with freely configurable otpodes. Example data here was taken from an open [Github Respository from Artinis](https://github.com/Artinis-Medical-Systems-B-V/snirf_data_example/tree/develop/single_device_finger_tapping)

In [ ]:
# Load the example NSP2 dataset (downloaded automatically on first use).
# To use your own data, replace this line with:
#   rec = cedalion.io.load_recording("/path/to/your/recording.snirf")
rec = cedalion.data.get_artinis_testdataset()

#### Step 1 — Channel statistics

Before looking at the signal, it is useful to characterise the channel geometry and raw signal quality.

- **Source–detector distance** determines measurement depth: short-separation channels (< 1 cm) primarily sample the scalp; long-separation channels (2.5–4.5 cm) are sensitive to cortical haemodynamics. Above 3.5 cm the sensitivity to cortical haemodynamics improves further, however the amount of received light and thus SNR drastically decreases. 
- **Signal-to-noise ratio (SNR)** is a fast first-pass quality check. Channels with very low SNR (typically caused by poor optode contact) should be excluded before further analysis.

> For a full quality assessment workflow — including scalp coupling index (SCI), peak-power ratio, and channel pruning — see the [signal quality notebook](../signal_quality/23_pruning.ipynb).

In [ ]:
# calculate the distances for all channels in the recording
distances = cedalion.nirs.channel_distances(rec["amp"], rec.geo3d)
channel_count = len(distances)

# calculate SNR for all channels
snr, _ = qlty.snr(rec["amp"])

# create the plots...
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
# ... a histogram of SNR ...
axes[0].hist(distances, bins=50, alpha=0.7, color="steelblue")
axes[0].set_title(f"Channel distances ({channel_count} channels)")
axes[0].set_xlabel("Distance (mm)")
axes[0].set_ylabel("Count")
axes[0].grid(True)
#... and a scatter plot of SNR vs distance
axes[1].scatter(distances, snr.sel(wavelength=805), alpha=0.7, color="coral", s=2)
axes[1].set_title("SNR vs Channel Distance")
axes[1].set_xlabel("Distance (mm)")
axes[1].set_ylabel("SNR")
axes[1].grid(True)
# show the plots
fig.tight_layout()
plt.show()


Depending on the measurement setup and vendor all possible source–detector combinations are stored in the SNIRF file, many of which can be too far apart to carry a useful cortical signal. For this example, we retain only channels with a source–detector distance below 3.5 cm.

In [ ]:
# generate the distance mask
ch_mask = cedalion.nirs.channel_distances(rec["amp"], rec.geo3d) < 3.5 * cedalion.units.cm

# apply the mask to the data
amp = rec["amp"].sel(channel=ch_mask)

# print the number of channels that pass the mask relative to the total number
print(f"Channels passing distance mask: {ch_mask.sum().item()} / {len(ch_mask)}")

#### Step 2 — Montage visualisation

Visualising the probe montage on a head surface confirms that the optodes are in anatomically plausible positions and gives an immediate sense of the cortical coverage.

We first plot the raw digitised positions reported in the SNIRF file, then register them to the Colin27 atlas scalp surface. Registration involves two sub-steps:

1. **Landmark normalisation** — map vendor-specific landmark labels (e.g. `"Nz"`, `"nz"`, `"nasion"`) to a common naming convention so they align with the atlas landmarks.
2. **Scalp snapping** — rigidly align the digitised positions to the atlas and project each optode onto the nearest point on the scalp mesh.

> To learn more about probe registration and co-registration using photogrammetry, see the [Photogrammetric Co-Registration Notebook](../head_models/41_photogrammetric_optode_coregistration.ipynb).

In [ ]:
# Quick 3-D view of the optode positions as stored in the SNIRF file,
# without any registration to a head model.
cedalion.vis.anatomy.plot_montage3D(amp, rec.geo3d)

In [ ]:
# Normalise landmark labels so they match the atlas convention, then
# rigidly align and snap the optode positions onto the Colin27 scalp surface.
geo3d = normalize_landmarks_labels(rec.geo3d)
geo3d_snapped = head_ras.align_and_snap_to_scalp(geo3d, mode="trans_rot_isoscale")

# Render the scalp (semi-transparent) and cortex together with the registered optodes.
plotter = pv.Plotter()
vbx.plot_surface(plotter, head_ras.scalp, opacity=0.1)
vbx.plot_surface(plotter, head_ras.brain, color="w")
vbx.plot_labeled_points(plotter, geo3d_snapped)
plotter.show()

#### Step 3 — Sensitivity profile

The sensitivity matrix (also called the Jacobian or *A* matrix) quantifies how much each cortical voxel contributes to each measured channel. Visualising the maximum sensitivity across all channels gives an intuitive map of the cortical area covered by the montage and reveals potential blind spots.

Computing the sensitivity matrix requires running a photon transport simulation (Monte Carlo or FEM), which is computationally expensive. Here we load a pre-computed sensitivity matrix for this dataset. To learn how to compute your own, see the [image reconstruction notebook](../head_models/40_image_reconstruction.ipynb).

In [ ]:
# Load a pre-computed sensitivity matrix for this dataset at the Colin27 atlas.
Adot = cedalion.data.get_precomputed_sensitivity("artinis_testdataset", "colin27")

# For a spatial overview, take the maximum sensitivity across all channels at 850 nm.
# Restricting to brain vertices avoids scalp contributions dominating the colour scale.
# Log10 is applied because sensitivity varies over several orders of magnitude with depth.
sensitivity = Adot.sel(vertex=Adot.is_brain, wavelength=805).max("channel")

plotter = pv.Plotter()
vbx.plot_surface(plotter, head_ras.scalp, color="w", opacity=0.1)
vbx.plot_surface(plotter, head_ras.brain, color=np.log10(np.clip(sensitivity, 1e-3, 1e1)), cmap="jet")
vbx.plot_labeled_points(plotter, geo3d_snapped)
plotter.show()

#### Step 4 — Time series

A quick look at the raw amplitude time series for a single channel confirms that the signal is present and physiologically plausible. Things to look for:

- A clear cardiac pulse (~1 Hz oscillation visible in a short window).
- No flat sections or sudden jumps that would indicate a saturated or disconnected optode.
- Stimulus markers aligned with the expected task timings.

> For detailed time series visualisation across many channels — including butterfly plots, quality overlays, and epoch-locked averages — see the [Visualization Examples Notebook](../plots_visualization/12_plots_example.ipynb).

In [ ]:
ch = amp.channel[0].item()  # pick first channel; replace with a specific channel name if desired

fig, ax = plt.subplots(figsize=(10, 3))
for wl, color in zip(amp.wavelength.values, ["b", "r", "c"]):
    ax.plot(amp.time, amp.sel(channel=ch, wavelength=wl), color=color, label=f"{wl} nm")
ax.set_title(f"Raw amplitude — Ch. {ch}")
ax.set_xlabel("Time / s")
ax.set_ylabel("Intensity / a.u.")
ax.set_xlim(0, 100)
ax.legend()
vbx.plot_stim_markers(ax, rec.stim, y=1)
plt.tight_layout()
plt.show()

---
### 4. Kernel

 [Kernel](https://www.kernel.com/) provides a wearable modular helmet, the "Flow", which is a high-density TD-fNIRS system designed for wearable, everyday use. Example data here is taken from the dataset published by [Dubois et al.](https://www.nature.com/articles/s41598-024-68555-9) on [OpenNeuro](https://openneuro.org/datasets/ds006545/versions/1.0.0).

In [ ]:
# Load the example NSP2 dataset (downloaded automatically on first use).
# To use your own data, replace this line with:
#   rec = cedalion.io.load_recording("/path/to/your/recording.snirf")
rec = cedalion.data.get_kernel_testdataset()

#### Step 1 — Channel statistics

Before looking at the signal, it is useful to characterise the channel geometry and raw signal quality.

- **Source–detector distance** determines measurement depth: short-separation channels (< 1 cm) primarily sample the scalp; long-separation channels (2.5–4.5 cm) are sensitive to cortical haemodynamics. Above 3.5 cm the sensitivity to cortical haemodynamics improves further, however the amount of received light and thus SNR drastically decreases. 
- **Signal-to-noise ratio (SNR)** is a fast first-pass quality check. Channels with very low SNR (typically caused by poor optode contact) should be excluded before further analysis.

> For a full quality assessment workflow — including scalp coupling index (SCI), peak-power ratio, and channel pruning — see the [signal quality notebook](../signal_quality/23_pruning.ipynb).

In [ ]:
# calculate the distances for all channels in the recording
distances = cedalion.nirs.channel_distances(rec["amp"], rec.geo3d)
channel_count = len(distances)

# calculate SNR for all channels
snr, _ = qlty.snr(rec["amp"])

# create the plots...
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
# ... a histogram of SNR ...
axes[0].hist(distances, bins=50, alpha=0.7, color="steelblue")
axes[0].set_title(f"Channel distances ({channel_count} channels)")
axes[0].set_xlabel("Distance (mm)")
axes[0].set_ylabel("Count")
axes[0].grid(True)
#... and a scatter plot of SNR vs distance
axes[1].scatter(distances, snr.sel(wavelength=905), alpha=0.7, color="coral", s=2)
axes[1].set_title("SNR vs Channel Distance")
axes[1].set_xlabel("Distance (mm)")
axes[1].set_ylabel("SNR")
axes[1].grid(True)
# show the plots
fig.tight_layout()
plt.show()


Depending on the measurement setup and vendor all possible source–detector combinations are stored in the SNIRF file, many of which can be too far apart to carry a useful cortical signal. For this example, we retain only channels with a source–detector distance below 3.5 cm.

In [ ]:
# generate the distance mask
ch_mask = cedalion.nirs.channel_distances(rec["amp"], rec.geo3d) < 3.5 * cedalion.units.cm

# apply the mask to the data
amp = rec["amp"].sel(channel=ch_mask)

# print the number of channels that pass the mask relative to the total number
print(f"Channels passing distance mask: {ch_mask.sum().item()} / {len(ch_mask)}")

#### Step 2 — Montage visualisation

Visualising the probe montage on a head surface confirms that the optodes are in anatomically plausible positions and gives an immediate sense of the cortical coverage.

We first plot the raw digitised positions reported in the SNIRF file, then register them to the Colin27 atlas scalp surface. Registration involves two sub-steps:

1. **Landmark normalisation** — map vendor-specific landmark labels (e.g. `"Nz"`, `"nz"`, `"nasion"`) to a common naming convention so they align with the atlas landmarks.
2. **Scalp snapping** — rigidly align the digitised positions to the atlas and project each optode onto the nearest point on the scalp mesh.

> To learn more about probe registration and co-registration using photogrammetry, see the [Photogrammetric Co-Registration Notebook](../head_models/41_photogrammetric_optode_coregistration.ipynb).

In [ ]:
# Quick 3-D view of the optode positions as stored in the SNIRF file,
# without any registration to a head model.
cedalion.vis.anatomy.plot_montage3D(amp, rec.geo3d)

In [ ]:
# Normalise landmark labels so they match the atlas convention, then
# rigidly align and snap the optode positions onto the Colin27 scalp surface.
geo3d = normalize_landmarks_labels(rec.geo3d)
geo3d_snapped = head_ras.align_and_snap_to_scalp(geo3d, mode="trans_rot_isoscale")

# Render the scalp (semi-transparent) and cortex together with the registered optodes.
plotter = pv.Plotter()
vbx.plot_surface(plotter, head_ras.scalp, opacity=0.1)
vbx.plot_surface(plotter, head_ras.brain, color="w")
vbx.plot_labeled_points(plotter, geo3d_snapped)
plotter.show()

#### Step 3 — Sensitivity profile

The sensitivity matrix (also called the Jacobian or *A* matrix) quantifies how much each cortical voxel contributes to each measured channel. Visualising the maximum sensitivity across all channels gives an intuitive map of the cortical area covered by the montage and reveals potential blind spots.

Computing the sensitivity matrix requires running a photon transport simulation (Monte Carlo or FEM), which is computationally expensive. Here we load a pre-computed sensitivity matrix for this dataset. To learn how to compute your own, see the [image reconstruction notebook](../head_models/40_image_reconstruction.ipynb).

In [ ]:
# Load a pre-computed sensitivity matrix for this dataset at the Colin27 atlas.
Adot = cedalion.data.get_precomputed_sensitivity("kernel_testdataset", "colin27")

# For a spatial overview, take the maximum sensitivity across all channels at 850 nm.
# Restricting to brain vertices avoids scalp contributions dominating the colour scale.
# Log10 is applied because sensitivity varies over several orders of magnitude with depth.
sensitivity = Adot.sel(vertex=Adot.is_brain, wavelength=905).max("channel")

plotter = pv.Plotter()
vbx.plot_surface(plotter, head_ras.scalp, color="w", opacity=0.1)
vbx.plot_surface(plotter, head_ras.brain, color=np.log10(np.clip(sensitivity, 1e-3, 1e1)), cmap="jet")
vbx.plot_labeled_points(plotter, geo3d_snapped)
plotter.show()

#### Step 4 — Time series

A quick look at the raw amplitude time series for a single channel confirms that the signal is present and physiologically plausible. Things to look for:

- A clear cardiac pulse (~1 Hz oscillation visible in a short window).
- No flat sections or sudden jumps that would indicate a saturated or disconnected optode.
- Stimulus markers aligned with the expected task timings.

> For detailed time series visualisation across many channels — including butterfly plots, quality overlays, and epoch-locked averages — see the [Visualization Examples Notebook](../plots_visualization/12_plots_example.ipynb).

In [ ]:
ch = amp.channel[0].item()  # pick first channel; replace with a specific channel name if desired

fig, ax = plt.subplots(figsize=(10, 3))
for wl, color in zip(amp.wavelength.values, ["b", "r"]):
    ax.plot(amp.time, amp.sel(channel=ch, wavelength=wl), color=color, label=f"{wl} nm")
ax.set_title(f"Raw amplitude — Ch. {ch}")
ax.set_xlabel("Time / s")
ax.set_ylabel("Intensity / a.u.")
ax.set_xlim(0, 100)
ax.legend()
vbx.plot_stim_markers(ax, rec.stim, y=1)
plt.tight_layout()
plt.show()